In [4]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [5]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained("facebook/bart-large-mnli")

In [6]:
print(model)

BartForSequenceClassification(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [7]:
model = model_obfuscation(model)

Obfuscating: model.encoder.layers.0.self_attn.k_proj
Obfuscating: model.encoder.layers.0.self_attn.v_proj
Obfuscating: model.encoder.layers.0.self_attn.q_proj
Obfuscating: model.encoder.layers.0.self_attn.out_proj
Obfuscating: model.encoder.layers.0.fc1
Obfuscating: model.encoder.layers.0.fc2
Obfuscating: model.encoder.layers.1.self_attn.k_proj
Obfuscating: model.encoder.layers.1.self_attn.v_proj
Obfuscating: model.encoder.layers.1.self_attn.q_proj
Obfuscating: model.encoder.layers.1.self_attn.out_proj
Obfuscating: model.encoder.layers.1.fc1
Obfuscating: model.encoder.layers.1.fc2
Obfuscating: model.encoder.layers.2.self_attn.k_proj
Obfuscating: model.encoder.layers.2.self_attn.v_proj
Obfuscating: model.encoder.layers.2.self_attn.q_proj
Obfuscating: model.encoder.layers.2.self_attn.out_proj
Obfuscating: model.encoder.layers.2.fc1
Obfuscating: model.encoder.layers.2.fc2
Obfuscating: model.encoder.layers.3.self_attn.k_proj
Obfuscating: model.encoder.layers.3.self_attn.v_proj
Obfuscating:

In [8]:
model = obfus_inference_mode(model)

In [9]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [10]:
# two examples from training set

sentence = "Conceptually cream skimming has two basic dimensions - product and geography. Product and geography are what make cream skimming work." # 2 neutral

print(classifier(sentence))

sentence = "you know during the season and i guess at at your level uh you lose them to the next level if if they decide to recall the the parent team the Braves decide to call to recall a guy from triple A then a double A guy goes up to replace him and a single A guy goes up to replace him. You lose the things to the following level if the people recall." # 1 entailment

print(classifier(sentence))

sentence = "At the end of Rue des Francs-Bourgeois is what many consider to be the city's most handsome residential square, the Place des Vosges, with its stone and red brick facades. Place des Vosges is constructed entirely of gray marble." # 0 contradiction
print(classifier(sentence))

[{'label': 'entailment', 'score': 0.9984042048454285}]
[{'label': 'entailment', 'score': 0.9984040856361389}]
[{'label': 'entailment', 'score': 0.9984040856361389}]


In [11]:
dataset = load_dataset("glue", "mnli")['validation_matched']
print(dataset)

Dataset({
    features: ['premise', 'hypothesis', 'label', 'idx'],
    num_rows: 9815
})


In [12]:
correct = 0
total = 0

for i in tqdm.tqdm(range(9815)):
    result = classifier(dataset[i]['premise'] + ' ' + dataset[i]['hypothesis'])

    if result[0]['label'] == 'neutral':
        result = 1
    elif result[0]['label'] == 'entailment':
        result = 0
    else:
        result = 2
        
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 9815/9815 [02:33<00:00, 63.99it/s]

correct:3479, total:9815, accuracy:0.3544574630667346
